In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = (
    SparkSession.builder.master("local[*]").appName("background_check").getOrCreate()
)


data = [
    (
        1,
        "John Doe",
        "1980-05-05",
        "Theft,Assault",
        "ABC Corp,XYZ Corp",
        "B.A. in English,M.A. in English",
        "123 Main St,456 Pine St",
    ),
    (
        2,
        "Jane Smith",
        "1982-07-12",
        "DUI",
        "DEF Corp,GHI Corp,JKL Corp",
        "B.Sc. in Physics",
        "789 Oak St",
    ),
    (
        3,
        "Bob Johnson",
        "1975-10-20",
        "Theft,Fraud,Embezzlement",
        "MNO Corp",
        "B.A. in History,M.A. in History",
        "321 Cedar St,654 Elm St",
    ),
]

schema = [
    "check_id",
    "full_name",
    "dob",
    "criminal_record",
    "employment_history",
    "education_history",
    "address",
]

background_checks = spark.createDataFrame(data, schema)
background_checks.show(truncate=False)

+--------+-----------+----------+------------------------+--------------------------+-------------------------------+-----------------------+
|check_id|full_name  |dob       |criminal_record         |employment_history        |education_history              |address                |
+--------+-----------+----------+------------------------+--------------------------+-------------------------------+-----------------------+
|1       |John Doe   |1980-05-05|Theft,Assault           |ABC Corp,XYZ Corp         |B.A. in English,M.A. in English|123 Main St,456 Pine St|
|2       |Jane Smith |1982-07-12|DUI                     |DEF Corp,GHI Corp,JKL Corp|B.Sc. in Physics               |789 Oak St             |
|3       |Bob Johnson|1975-10-20|Theft,Fraud,Embezzlement|MNO Corp                  |B.A. in History,M.A. in History|321 Cedar St,654 Elm St|
+--------+-----------+----------+------------------------+--------------------------+-------------------------------+-----------------------+



In [4]:
count_items = udf(
    lambda s: len(s.split(",")) if s else 0,
    IntegerType(),
)

# processing each column
result_df = (
    background_checks.withColumn(
        "crime_count",
        count_items(col("criminal_record")),
    )
    .withColumn(
        "jobs_count",
        count_items(col("employment_history")),
    )
    .withColumn(
        "degrees_count",
        count_items(col("education_history")),
    )
    .withColumn(
        "places_lived_count",
        count_items(col("address")),
    )
)

# selecting only the required columns
result_df = result_df.select(
    "check_id",
    "full_name",
    "dob",
    "crime_count",
    "jobs_count",
    "degrees_count",
    "places_lived_count",
)
result_df.show()

+--------+-----------+----------+-----------+----------+-------------+------------------+
|check_id|  full_name|       dob|crime_count|jobs_count|degrees_count|places_lived_count|
+--------+-----------+----------+-----------+----------+-------------+------------------+
|       1|   John Doe|1980-05-05|          2|         2|            2|                 2|
|       2| Jane Smith|1982-07-12|          1|         3|            1|                 1|
|       3|Bob Johnson|1975-10-20|          3|         1|            2|                 2|
+--------+-----------+----------+-----------+----------+-------------+------------------+

